# Cleaning crawlers out of the imported Umami data

`05_traffic_bursts.ipynb` found that ~40.6% of the imported Umami history (655 of 1612
pageviews) is two crawler signatures, not real visitors. This applies that finding: recompute
the affected `rollup/{site}/{YYYY-MM}.json` objects from the same CSV with those rows
excluded, and overwrite.

**Why this can only touch the Umami era.** The live beacon (`hit.ts`) has only ever stored
`{hits, pages}` — no browser, screen, referrer, session, or country. Once a visit is
aggregated into that shape there's nothing left to filter. Only the raw CSV export still has
the signals `05_traffic_bursts.ipynb` used, so cleaning can only mean: re-aggregate the CSV,
overwrite the `source: "umami"` objects it produced. Nothing from the beacon era can be
touched this way — `analytics/hit.ts` and `website/utils/hitTracker.ts` (Tier 1, shipped
separately) stop new crawler traffic from being recorded going forward; this notebook only
corrects what's already there.

**The real scope is Jan 1 – Aug 9, not the full CSV range.** The CSV runs through 2026-08-10,
but that's a partial day (3 rows, last at 05:12 UTC) that real, uncontaminated beacon data
already covers (two hourly buckets already sit under `counts/fretchen.eu/2026-08-10T*.json`).
Aug 10 doesn't need cleaning — it needs *removing* from the rollup so the system's own
existing fallback (`collectRange` in `stats.ts`, the weekly `rollup` cron) picks up the real
data it already has, as `source: "beacon"`.

**`source` stays `"umami"`.** Cleaning improves data *within* the historical category; it
doesn't create a new one. Changing the label would need a matching change to
`website/utils/analyticsBuckets.ts`'s `historic` flag — out of scope for a data correction.

In [1]:
import os

from dotenv import load_dotenv

from storage import S3Storage
from umami_backfill import SITE, read_pageviews, to_monthly_rollups

load_dotenv()  # searches upward — finds ../.env (analytics/.env)

CSV_PATH = "export/website_event.csv"
CUTOFF = "2026-08-10"  # exclusive — the true Umami-only era ends the day before this

storage = S3Storage(
    access_key=os.environ["SCW_ACCESS_KEY"],
    secret_key=os.environ["SCW_SECRET_KEY"],
)

## 1. Recompute with crawlers excluded, and cross-check against notebook 05

`exclude_crawlers=True` applies the two signatures from `05_traffic_bursts.ipynb`
(`is_chronic_crawler`, `find_acute_crawler_session_ids`) before rows are reduced to
`(day, path)`. The removed count is asserted against that notebook's already-established
figure — if the CSV or the logic ever drifts, this fails loudly instead of silently
producing a different cleanup than what was actually analysed.

In [2]:
before = read_pageviews(CSV_PATH)
after = read_pageviews(CSV_PATH, exclude_crawlers=True)
removed = len(before) - len(after)

print(f"before:  {len(before)}")
print(f"after:   {len(after)}")
print(f"removed: {removed}  ({removed / len(before):.1%})")

assert removed == 655, f"expected 655 removed (per notebook 05), got {removed}"

before:  1612
after:   957
removed: 655  (40.6%)


## 2. Cap to the true Umami-only era

`2026-08-10` is excluded here — handled separately in section 4, not recomputed.

In [3]:
before_capped = [(day, path) for day, path in before if day < CUTOFF]
after_capped = [(day, path) for day, path in after if day < CUTOFF]

before_rollups = to_monthly_rollups(before_capped)
cleaned_rollups = to_monthly_rollups(after_capped)

print(f"{'month':<10} {'before':>8} {'after':>8} {'removed':>8}")
for month in sorted(before_rollups):
    before_hits = sum(d["hits"] for d in before_rollups[month]["days"].values())
    after_hits = sum(d["hits"] for d in cleaned_rollups.get(month, {"days": {}})["days"].values())
    print(f"{month:<10} {before_hits:>8} {after_hits:>8} {before_hits - after_hits:>8}")

month        before    after  removed
2026-01         199       93      106
2026-02         184      107       77
2026-03         315      183      132
2026-04         202      146       56
2026-05         217      158       59
2026-06         203      123       80
2026-07         155      119       36
2026-08         134       27      107


## 3. Full months (Jan–Jul) — direct overwrite, no merge

`merge_into_existing`'s "existing day always wins" rule exists to protect the *original*
incremental backfill from clobbering good data on a re-run — the wrong tool for a deliberate
correction. These seven months contain only Umami-sourced days (the beacon didn't exist yet),
so a wholesale overwrite is safe and simpler than merging.

In [4]:
WRITE_TO_S3 = True  # flip to True to write for real

full_months = [m for m in cleaned_rollups if m != "2026-08"]
print(f"{len(full_months)} full months to overwrite: {full_months}")

if WRITE_TO_S3:
    for month in sorted(full_months):
        storage.write(f"rollup/{SITE}/{month}.json", cleaned_rollups[month])
    print(f"wrote {len(full_months)} objects")

7 full months to overwrite: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']


wrote 7 objects


## 4. August — read-modify-write, not overwrite

The existing August rollup holds a mix of `source: "umami"` days (01-10) and real
`source: "beacon"` days (11 onward, already compacted by the live `/stats` write-back or the
weekly cron) — overwriting the whole object would destroy that real data. Instead: keep every
day that isn't `"umami"`-sourced untouched, and replace the umami days with the freshly
cleaned set. Since the cleaned set is capped before `CUTOFF`, day 10 is simply absent from
it — dropped, not recomputed, exactly as section 2 above.

In [5]:
existing_aug = storage.read(f"rollup/{SITE}/2026-08.json") or {
    "site": SITE,
    "month": "2026-08",
    "days": {},
}
preserved_days = {day: b for day, b in existing_aug["days"].items() if b["source"] != "umami"}
cleaned_aug_days = cleaned_rollups.get("2026-08", {"days": {}})["days"]

print(f"preserved (non-umami) days: {sorted(preserved_days)}")
print(f"cleaned umami days (01-09): {sorted(cleaned_aug_days)}")
assert "2026-08-10" not in cleaned_aug_days, "cutoff should have excluded Aug 10"

new_aug = {
    "site": SITE,
    "month": "2026-08",
    "days": dict(sorted({**preserved_days, **cleaned_aug_days}.items())),
}

if WRITE_TO_S3:
    storage.write(f"rollup/{SITE}/2026-08.json", new_aug)
    print("wrote rollup/fretchen.eu/2026-08.json")

preserved (non-umami) days: ['2026-08-11', '2026-08-14']
cleaned umami days (01-09): ['2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05', '2026-08-07', '2026-08-09']


wrote rollup/fretchen.eu/2026-08.json


## 5. Read back and confirm

Only meaningful after a real write (`WRITE_TO_S3 = True`).

In [6]:
if WRITE_TO_S3:
    for month in sorted(cleaned_rollups):
        stored = storage.read(f"rollup/{SITE}/{month}.json")
        n_days = len(stored["days"])
        hits = sum(d["hits"] for d in stored["days"].values())
        sources = sorted({d["source"] for d in stored["days"].values()})
        has_aug_10 = "2026-08-10" in stored["days"]
        print(f"{month}: {n_days} days, {hits} hits, sources={sources}, aug10={has_aug_10}")

2026-01: 19 days, 93 hits, sources=['umami'], aug10=False


2026-02: 15 days, 107 hits, sources=['umami'], aug10=False


2026-03: 26 days, 183 hits, sources=['umami'], aug10=False
2026-04: 24 days, 146 hits, sources=['umami'], aug10=False


2026-05: 27 days, 158 hits, sources=['umami'], aug10=False
2026-06: 26 days, 123 hits, sources=['umami'], aug10=False
2026-07: 25 days, 119 hits, sources=['umami'], aug10=False


2026-08: 9 days, 49 hits, sources=['beacon', 'umami'], aug10=False
